# PosterLlama + HyperCLOVA X — GPU 본 학습

**이 노트북은 실제 사전학습 가중치와 사용자 데이터로 2단계 학습을 실행합니다.** 서버로 수정된 노트북을 복사한 뒤 **커널을 재시작**하고, 다운로드 셀의 DATA_ROOT를 설정한 다음 위에서부터 실행하세요. 기존 CPU 테스트 모델의 변수는 재사용하지 마세요.

| 단계 | 이미지 인코더 | Adapter | CodeLlama | 목표 |
|---|---|---|---|---|
| 1 | 동결 | 학습 | 동결 | 이미지 → 자연어 설명 |
| 2 | 동결 | 1단계 가중치 동결 | 기본 가중치 동결 + LoRA 학습 | 이미지 + 지시문 → HTML 레이아웃 |

- 비전: `naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B`의 SigLIP 타워
- 언어 모델: `codellama/CodeLlama-7b-hf`
- `smoke=False`, CUDA GPU 1개 사용, 합성 데이터 생성 없음, step 제한 없음
- 기본 시작값: **1단계 3에폭 / 2단계 10에폭**, 배치 1, gradient accumulation 8, LoRA rank 64. 데이터 규모와 검증 결과에 따라 조정하세요. 이 횟수가 수렴이나 논문 성능을 보장하지 않습니다.

공개 HyperCLOVA 모델의 언어 모델·AnyRes·C-Abstractor는 사용하지 않습니다. [공식 모델 설정](https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B/blob/main/config.json)을 참고한 비전 타워 이식 구현입니다.

## 0. GPU 서버 커널과 패키지

서버의 Python **3.10+** 커널을 선택하세요. 아래 설치 명령은 필요한 경우에만 주석을 해제하고 실행한 뒤 커널을 재시작하세요. CUDA 지원 PyTorch가 필요하며, GPU 인식에 실패하면 CPU 학습으로 자동 전환하지 않습니다.

기본은 비양자화 학습입니다. GPU 메모리가 부족하면 다음 설정 셀의 `LOAD_IN_4BIT=True`와 `bitsandbytes` 설치를 사용할 수 있습니다. 이 옵션은 양자화된 CodeLlama에 Adapter/LoRA를 학습하는 변형입니다. 여러 GPU가 보이더라도 이 노트북은 선택한 GPU 1개만 사용합니다.

In [1]:
import sys
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10+ 서버 커널이 필요합니다: {sys.version}")
print("Python:", sys.executable)
# %pip install -r requirements.txt
# %pip install bitsandbytes  # LOAD_IN_4BIT=True일 때 추가 설치

Python: /opt/homebrew/anaconda3/envs/naver/bin/python


In [ ]:
import gc
import json
import math
import importlib.util
from datetime import datetime

import torch
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "posterllama_hcx").is_dir():
    raise RuntimeError("노트북 작업 디렉터리를 서버의 PosterLlama 프로젝트 폴더로 설정하세요.")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU가 인식되지 않습니다. 서버 터미널에서 nvidia-smi를 확인하고 "
        "CUDA 지원 PyTorch가 설치된 서버 커널을 선택하세요. 본 학습을 CPU로 실행하지 않습니다."
    )

from posterllama_hcx import (
    ModelConfig, TrainConfig, PosterLlama, PosterDataset,
    train_stage, evaluate, build_layout_prompt, render_layout,
)
from posterllama_hcx.training import seed_everything

GPU_INDEX = 0  # CUDA_VISIBLE_DEVICES 적용 후 보이는 GPU 번호
if not 0 <= GPU_INDEX < torch.cuda.device_count():
    raise ValueError("GPU_INDEX가 사용 가능한 GPU 범위를 벗어났습니다.")
torch.cuda.set_device(GPU_INDEX)
DEVICE = f"cuda:{GPU_INDEX}"
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
free_bytes, total_bytes = torch.cuda.mem_get_info(GPU_INDEX)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(GPU_INDEX), "선택 장치:", DEVICE)
print(f"GPU 메모리: 전체 {total_bytes / 2**30:.1f} GiB / 여유 {free_bytes / 2**30:.1f} GiB")
print("학습 정밀도:", DTYPE)

## 0.1. 이미지와 정답 다운로드 · 압축 해제 · 데이터 연결

**처음에는 이 셀에서 실제 학습 데이터를 받습니다.** `DATA_ROOT`를 GPU 서버의 여유 있는 디스크로 지정하세요. 첫 실행의 기본 여유 공간 검사값은 100 GiB이며 실제 압축 해제 공간도 별도로 검사합니다. 압축파일은 재실행을 위해 보관합니다. 모델 캐시가 다른 디스크에 있다면 그 디스크의 공간도 확인하세요.

| 용도 | 다운로드 | 내용 |
|---|---|---|
| 1단계 시작용 캡션 | MiniGPT-4 `cc_sbu_align.zip` (약 29 MB) | 실제 이미지·설명 3,439쌍, 이미지 단위 95%/5% 분할 |
| 2단계 HTML 정답 | PosterLlama `html_format_img_instruct_mask_all_condition.zip` (약 49 MB) | 원본 학습·검증 JSONL, 압축 해제 약 2.5 GB |
| 2단계 포스터 이미지 | PosterLlama `cgl_inpainting_all.zip` (약 28.2 GB) | 글자를 지운 포스터 배경 이미지 |

1단계 `cc_sbu_align`은 **MiniGPT-4의 소규모 정렬 세트**를 이번 Adapter 초기 학습에 사용한 것입니다. PosterLlama의 원래 1단계 LAION/CC/SBU 전체 데이터(준비 용량 약 2.3 TB)를 재현하는 설정은 아닙니다. 3,439쌍만으로 충분한 일반화 성능을 보장하지 않습니다. 원본 2단계 일반 레이아웃 데이터에는 실제 홍보 문구 조건이 없는 작업도 있으므로, 한국어 문구 조건 성능에는 별도 데이터가 필요합니다.

출처: [MiniGPT-4 소규모 세트](https://github.com/Vision-CAIR/MiniGPT-4/blob/main/dataset/README_2_STAGE.md), [원본 1단계 규모](https://github.com/Vision-CAIR/MiniGPT-4/blob/main/dataset/README_1_STAGE.md), [PosterLlama 저자 공개 데이터](https://drive.google.com/drive/folders/1OK7O4h_JxaEVzm5Jlb2YrsKjwYmHfKNv).

파일이 있으면 재사용합니다. 다운로드 중단 시 같은 셀을 재실행해 이어받을 수 있습니다. Drive 할당량/권한 오류는 메시지와 원본 링크를 표시합니다. 작은 두 ZIP은 직접 받은 SHA256으로, 큰 이미지 ZIP은 크기와 압축 해제 CRC로 검증합니다. 원본이 갱신되어 크기/해시가 달라지면 자동으로 통과시키지 않습니다.

이 노트북을 열어 두고 실행을 완료하세요. 이 셀은 데이터 다운로드만 수행하며 아직 모델 학습은 시작하지 않습니다.

In [ ]:
import hashlib
import random
import shutil
import stat
import subprocess
import zipfile
from pathlib import PurePosixPath

# GPU 서버에 충분한 공간이 있는 경로를 지정하세요. 예: Path('/data/PosterLlama')
DATA_ROOT = PROJECT_ROOT / "data"
DOWNLOAD_DATA = True
MIN_FREE_GIB = 100  # 원본 이미지 zip + 압축 해제 + 모델 캐시용 보수적 여유 공간

SOURCES = {
    "captions": {
        "id": "1nJXhoEcy3KTExr17I7BXqY5Y9Lx_-n-9",
        "filename": "cc_sbu_align.zip", "bytes": 29208145,
        "sha256": "db338e75f9d101b35d8787464a83182465b5abc86bc817efee0466d71fd5b3e4",
    },
    "layouts": {
        "id": "1M5mWTboGeLf-6MxKHH2ddaH6wyeMdeeG",
        "filename": "html_format_img_instruct_mask_all_condition.zip", "bytes": 48777076,
        "sha256": "7c3657a44ebae7d93ebad165208c8a95366feb47d6b7271bee3a66bea8eafc95",
    },
    "posters": {
        "id": "1lZz1DyRTKGo79xzOzMO56iKlzAEQUtvq",
        "filename": "cgl_inpainting_all.zip", "bytes": 28212617946,
        "sha256": None,  # 공개 폴더의 크기 확인값; 전체 다운로드/CRC 검증은 실행 시 수행
    },
}


def download_archive(source, directory):
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / source["filename"]
    if not target.exists():
        if importlib.util.find_spec("gdown") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown==5.2.0"])
            importlib.invalidate_caches()
        import gdown
        try:
            result = gdown.download(id=source["id"], output=str(target),
                                    quiet=False, resume=True, use_cookies=False)
        except Exception as error:
            raise RuntimeError(
                f"다운로드 실패: {source['filename']}. Drive 할당량/네트워크 오류라면 나중에 같은 셀을 재실행하세요. "
                f"직접 받은 파일을 {target}에 놓아도 됩니다. https://drive.google.com/file/d/{source['id']}/view"
            ) from error
        if not result:
            raise RuntimeError(f"다운로드가 완료되지 않았습니다: {target}")
    if target.stat().st_size != source["bytes"]:
        raise ValueError(f"파일 크기가 다릅니다: {target}. 불완전한 파일 또는 원본 변경 여부를 확인하세요.")
    if source["sha256"]:
        digest = hashlib.sha256()
        with target.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        if digest.hexdigest() != source["sha256"]:
            raise ValueError(f"SHA256 검증 실패: {target}")
    if not zipfile.is_zipfile(target):
        raise ValueError(f"정상 ZIP 파일이 아닙니다: {target}")
    return target


def extract_archive(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    destination = destination.resolve()
    marker = destination / ".extracted.json"
    signature = {"archive": archive.name, "bytes": archive.stat().st_size}
    if marker.is_file() and json.loads(marker.read_text()) == signature:
        return destination
    with zipfile.ZipFile(archive) as bundle:
        entries = bundle.infolist()
        for entry in entries:
            relative = PurePosixPath(entry.filename)
            if relative.is_absolute() or ".." in relative.parts or "\\" in entry.filename:
                raise ValueError(f"압축파일 내부 경로가 잘못되었습니다: {entry.filename}")
            if stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("데이터 ZIP의 symbolic link는 지원하지 않습니다.")
            if not (destination / entry.filename).resolve().is_relative_to(destination):
                raise ValueError("압축 해제 경로가 데이터 디렉터리를 벗어납니다.")
        required_bytes = sum(entry.file_size for entry in entries)
        if shutil.disk_usage(destination).free < required_bytes + 5 * 2**30:
            raise OSError(f"압축 해제 공간이 부족합니다. 추가로 약 {required_bytes / 2**30 + 5:.1f} GiB를 확보하세요.")
        from tqdm.auto import tqdm
        for entry in tqdm(entries, desc=f"압축 해제: {archive.name}"):
            output = destination / entry.filename
            if entry.is_dir():
                output.mkdir(parents=True, exist_ok=True)
                continue
            output.parent.mkdir(parents=True, exist_ok=True)
            # ZipExtFile을 끝까지 읽으면 CRC도 검증됩니다. 중단 시 marker가 없어 다시 해제합니다.
            with bundle.open(entry) as src, output.open("wb") as dst:
                shutil.copyfileobj(src, dst, length=1024 * 1024)
    marker.write_text(json.dumps(signature), encoding="utf-8")
    return destination


def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".jsonl.tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    temporary.replace(path)


def prepare_caption_manifest(extracted, data_root):
    candidates = list(extracted.rglob("filter_cap.json"))
    if len(candidates) != 1:
        raise ValueError("cc_sbu_align/filter_cap.json을 정확히 하나 찾을 수 있어야 합니다.")
    annotation_file = candidates[0]
    annotations = json.loads(annotation_file.read_text())["annotations"]
    image_root = annotation_file.parent / "image"
    rows, ids = [], set()
    for row in annotations:
        image_id = str(row["image_id"])
        if image_id in ids:
            raise ValueError(f"중복 caption image_id: {image_id}")
        ids.add(image_id)
        image_path = image_root / f"{image_id}.jpg"
        if not image_path.is_file() or not row["caption"].strip():
            raise ValueError(f"이미지 또는 캡션이 누락되었습니다: {image_id}")
        with Image.open(image_path) as image:
            image.verify()
        rows.append({"image": str(image_path.relative_to(data_root)), "caption": row["caption"]})
    random.Random(42).shuffle(rows)
    val_count = max(1, round(len(rows) * 0.05))
    if len(rows) <= val_count:
        raise ValueError("학습/검증으로 나눌 캡션 샘플이 부족합니다.")
    train_path, val_path = data_root / "stage1/train.jsonl", data_root / "stage1/val.jsonl"
    write_jsonl(train_path, rows[val_count:])
    write_jsonl(val_path, rows[:val_count])
    print(f"1단계 실제 이미지: train={len(rows) - val_count:,}, val={val_count:,}")
    return {"stage1_train": str(train_path), "stage1_val": str(val_path), "stage1_image_root": str(data_root)}


def find_unique(directory, filename):
    found = list(directory.rglob(filename))
    if len(found) != 1:
        raise ValueError(f"{filename} 파일을 정확히 하나 찾을 수 있어야 합니다: {found}")
    return found[0]


def resolve_image(image_root, name):
    path = image_root / name
    for candidate in [path] + [path.with_suffix(suffix) for suffix in (".png", ".jpg", ".jpeg")]:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(path)


def prepare_layout_paths(layout_root, poster_root):
    train_path = find_unique(layout_root, "train_llama_numerical.jsonl")
    val_path = find_unique(layout_root, "val_llama_numerical.jsonl")
    with train_path.open(encoding="utf-8") as handle:
        first = json.loads(next(handle))["name"]
    first = first[0] if isinstance(first, list) else first
    candidates = []
    for suffix in (Path(first).suffix, ".png", ".jpg", ".jpeg"):
        candidates.extend(poster_root.rglob(Path(first).with_suffix(suffix).name))
    roots = {path.parent for path in candidates}
    if len(roots) != 1:
        raise ValueError(f"CGL 이미지 루트를 특정할 수 없습니다: {roots}")
    image_root = roots.pop()
    seen = {}
    for split, path in (("train", train_path), ("val", val_path)):
        image_ids, missing, count = set(), [], 0
        with path.open(encoding="utf-8") as handle:
            for count, line in enumerate(handle, 1):
                row = json.loads(line)
                names = row["name"]
                name = names[0] if isinstance(names, list) else names
                # 기본 augment=False이므로 원본 이미지를 검사합니다. 증강 목록은 유지합니다.
                if name not in image_ids:
                    try:
                        resolve_image(image_root, name)
                    except FileNotFoundError:
                        missing.append(name)
                image_ids.add(name)
                task_values = [v for k, v in row.items() if k.endswith("_seq_modeling")]
                if not task_values or any(not isinstance(v, str) or "<MID>" not in v for v in task_values):
                    raise ValueError(f"2단계 {split} 레코드 {count}의 작업/정답 형식이 잘못되었습니다.")
                if count % 50000 == 0:
                    print(f"2단계 {split}: {count:,} 레코드 검사 중...", flush=True)
        if missing:
            raise FileNotFoundError(f"2단계 {split}: 이미지 {len(missing):,}개 누락. 예: {missing[:5]}")
        seen[split] = image_ids
        print(f"2단계 {split}: {count:,} 레코드 / 실제 이미지 {len(image_ids):,}개 경로 확인")
    overlap = seen["train"] & seen["val"]
    if overlap:
        raise ValueError(f"2단계 train/val에 동일 이미지 {len(overlap)}개가 겹칩니다.")
    return {"stage2_train": str(train_path), "stage2_val": str(val_path), "stage2_image_root": str(image_root)}


DATA_ROOT = DATA_ROOT.expanduser().resolve()
DATA_ROOT.mkdir(parents=True, exist_ok=True)
prepared_file = DATA_ROOT / "prepared_paths.json"
if DOWNLOAD_DATA:
    downloads = DATA_ROOT / "downloads"
    poster_marker = DATA_ROOT / "raw/posters/.extracted.json"
    # 준비 완료 이후 재실행은 대형 다운로드/재해제를 생략합니다.
    if not poster_marker.is_file() and shutil.disk_usage(DATA_ROOT).free < MIN_FREE_GIB * 2**30:
        raise OSError(
            f"첫 다운로드에는 여유 공간 {MIN_FREE_GIB} GiB를 권장합니다. 현재 "
            f"{shutil.disk_usage(DATA_ROOT).free / 2**30:.1f} GiB. "
            "DATA_ROOT를 GPU 서버의 여유 있는 데이터 디스크로 지정하세요."
        )
    print("데이터 다운로드 위치:", DATA_ROOT)
    roots = {}
    for name, source in SOURCES.items():
        archive = download_archive(source, downloads)
        roots[name] = extract_archive(archive, DATA_ROOT / "raw" / name)
    DATASET_PATHS = prepare_caption_manifest(roots["captions"], DATA_ROOT)
    DATASET_PATHS.update(prepare_layout_paths(roots["layouts"], roots["posters"]))
    prepared_file.write_text(json.dumps(DATASET_PATHS, ensure_ascii=False, indent=2), encoding="utf-8")
    (DATA_ROOT / "sources.json").write_text(json.dumps(SOURCES, indent=2), encoding="utf-8")
    print("이미지·정답 준비 완료:", prepared_file)
else:
    # 이미 직접 준비한 데이터는 다음 셀에서 경로를 직접 지정할 수도 있습니다.
    DATASET_PATHS = json.loads(prepared_file.read_text()) if prepared_file.is_file() else {}


## 1. 학습 설정과 실제 데이터 경로

1단계 JSONL 한 줄: `{"image": "images/a.jpg", "caption": "제품 이미지의 정답 설명"}`

2단계 JSONL 한 줄: `{"image": "images/a.jpg", "prompt": "작업 지시와 홍보 문구", "html": "<html><body><svg ...><rect .../></svg></body></html>"}`

- `STAGE1_IMAGE_ROOT`, `STAGE2_IMAGE_ROOT`는 각각 해당 JSONL 안의 이미지 경로 앞에 붙일 폴더입니다. `None`이면 JSONL의 부모 폴더를 사용합니다. 검증 이미지 폴더도 별도로 지정할 수 있습니다.
- `STAGE1_VAL_JSONL`, `STAGE2_VAL_JSONL`에 분리된 검증 파일을 지정하면 매 에폭 validation loss를 기록합니다. `None`이면 검증을 생략하며 학습 loss만으로 성능을 판단하지 마세요.
- 원본 논문의 2단계 `name`, `*_seq_modeling` + `<MID>` 포맷을 지원합니다. `PAPER_TASK="random"`은 그 포맷에서 7개 작업 중 존재하는 작업을 샘플링합니다. 직접 작성한 `prompt/html` 레코드에는 영향을 주지 않습니다.
- 원본 1단계 WebDataset tar는 이 로더로 직접 읽을 수 없습니다. 먼저 이미지·캡션 JSONL로 준비하세요.
- **앞의 다운로드 셀을 실행하면 실제 이미지와 정답을 받아 아래 경로에 자동 연결합니다.** 직접 준비한 데이터를 쓰려면 DOWNLOAD_DATA=False로 두고 아래 경로를 수정하세요.

매 실행마다 별도의 출력 폴더를 생성합니다. 기존 HCX 1단계 가중치를 재사용하려면 `TRAIN_STAGE1=False`로 바꾸고 `REUSE_STAGE1_CHECKPOINT`에 실제 체크포인트 폴더를 넣으세요. 소형 테스트/DINO 체크포인트는 사용할 수 없습니다.

In [ ]:
MODE = "real"
LOAD_IN_4BIT = False  # 메모리가 부족하면 True; bitsandbytes 설치 필요

# 서버의 실제 파일 경로로 바꾸세요.
STAGE1_JSONL = Path(DATASET_PATHS.get("stage1_train", DATA_ROOT / "stage1/train.jsonl"))
STAGE2_JSONL = Path(DATASET_PATHS.get("stage2_train", DATA_ROOT / "stage2/train.jsonl"))
STAGE1_IMAGE_ROOT = DATASET_PATHS.get("stage1_image_root")
STAGE2_IMAGE_ROOT = DATASET_PATHS.get("stage2_image_root")
STAGE1_VAL_JSONL = DATASET_PATHS.get("stage1_val")
STAGE2_VAL_JSONL = DATASET_PATHS.get("stage2_val")
STAGE1_VAL_IMAGE_ROOT = STAGE1_IMAGE_ROOT
STAGE2_VAL_IMAGE_ROOT = STAGE2_IMAGE_ROOT
PAPER_TASK = "random"

TRAIN_STAGE1 = True
REUSE_STAGE1_CHECKPOINT = None  # TRAIN_STAGE1=False일 때 실제 HCX stage1 폴더 지정
STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 10
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
STAGE1_LR = 1e-4
STAGE2_LR = 1e-4
MAX_PROMPT_TOKENS = 1024
MAX_TARGET_TOKENS = 1024
SEED = 42

stage1_train_config = TrainConfig(
    epochs=STAGE1_EPOCHS, batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=STAGE1_LR, max_steps=None, seed=SEED,
)
stage2_train_config = TrainConfig(
    epochs=STAGE2_EPOCHS, batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=STAGE2_LR, max_steps=None, seed=SEED,
)

if LOAD_IN_4BIT and importlib.util.find_spec("bitsandbytes") is None:
    raise ImportError("LOAD_IN_4BIT=True에는 bitsandbytes가 필요합니다. 설치 후 커널을 재시작하세요.")

# 파일 누락을 모델 다운로드 전에 한 번에 알려줍니다.
required = {"STAGE2_JSONL": STAGE2_JSONL}
if TRAIN_STAGE1:
    required["STAGE1_JSONL"] = STAGE1_JSONL
elif REUSE_STAGE1_CHECKPOINT is None:
    raise ValueError("TRAIN_STAGE1=False이면 REUSE_STAGE1_CHECKPOINT를 지정하세요.")
for name, path in (("STAGE1_VAL_JSONL", STAGE1_VAL_JSONL), ("STAGE2_VAL_JSONL", STAGE2_VAL_JSONL)):
    if path is not None:
        required[name] = path
missing = [f"{name}: {path}" for name, path in required.items() if not Path(path).expanduser().is_file()]
if missing:
    raise FileNotFoundError(
        "실제 학습 데이터 파일을 준비하고 이 셀의 경로를 수정하세요.\n"
        + "\n".join(missing)
        + "\n데이터가 없으면 본 학습을 시작할 수 없습니다."
    )

if not TRAIN_STAGE1:
    REUSE_STAGE1_CHECKPOINT = Path(REUSE_STAGE1_CHECKPOINT).expanduser()
    saved = json.loads((REUSE_STAGE1_CHECKPOINT / "posterllama.json").read_text())
    if saved["stage"] != 1 or not saved["adapter_ready"] or saved["model_config"]["smoke"]:
        raise ValueError("실제 HCX 모델로 학습한 1단계 Adapter 체크포인트가 필요합니다.")

stage1_data = PosterDataset(STAGE1_JSONL, 1, image_root=STAGE1_IMAGE_ROOT) if TRAIN_STAGE1 else None
stage2_data = PosterDataset(STAGE2_JSONL, 2, image_root=STAGE2_IMAGE_ROOT, task=PAPER_TASK)
val1 = PosterDataset(STAGE1_VAL_JSONL, 1, image_root=STAGE1_VAL_IMAGE_ROOT) if STAGE1_VAL_JSONL else None
val2 = PosterDataset(STAGE2_VAL_JSONL, 2, image_root=STAGE2_VAL_IMAGE_ROOT, task="unconditional") if STAGE2_VAL_JSONL else None

# 이미지/스키마는 각 데이터셋의 처음 최대 8개 샘플을 사전 점검합니다.
# 전체 레코드 점검은 아니며 나머지 레코드의 오류는 학습 중에 보고됩니다.
for name, dataset in (("stage1 train", stage1_data), ("stage2 train", stage2_data), ("stage1 val", val1), ("stage2 val", val2)):
    if dataset is not None:
        for index in range(min(8, len(dataset))):
            _ = dataset[index]
        print(f"{name}: {len(dataset):,} samples")
if val1 is None or val2 is None:
    print("검증 파일을 지정하지 않은 단계는 validation loss를 계산하지 않습니다.")

seed_everything(SEED)
example = (val2 if val2 is not None else stage2_data)[0]
EXAMPLE_SOURCE = "검증 데이터" if val2 is not None else "학습 데이터 (성능 평가용 아님)"
display(example["image"])
print("추론 예제 출처:", EXAMPLE_SOURCE)
print("입력 지시문:", example["prompt"])

RUN_DIR = PROJECT_ROOT / "outputs" / "hcx_codellama" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
STAGE1_CHECKPOINT = RUN_DIR / "stage1" if TRAIN_STAGE1 else REUSE_STAGE1_CHECKPOINT
STAGE2_CHECKPOINT = RUN_DIR / "stage2"
for name, dataset, cfg in (("stage1", stage1_data, stage1_train_config), ("stage2", stage2_data, stage2_train_config)):
    if dataset is not None:
        steps = math.ceil(math.ceil(len(dataset) / cfg.batch_size) / cfg.gradient_accumulation_steps) * cfg.epochs
        print(f"{name}: {cfg.epochs} epochs / 총 {steps:,} optimizer steps / max_steps=None")
print("유효 배치 크기:", BATCH_SIZE * GRADIENT_ACCUMULATION, "(마지막 그룹은 더 작을 수 있음)")
print("이번 실행 저장 경로:", RUN_DIR)

## 2. 실제 HyperCLOVA 비전 타워 + CodeLlama 로드

첫 실행에는 HCX 비전 가중치가 포함된 shard와 CodeLlama 가중치를 다운로드합니다. 모델 접근 인증이 필요하면 서버 터미널에서 `huggingface-cli login`을 먼저 실행하세요. 다운로드 시간과 CPU RAM/디스크 공간도 필요합니다.

- `smoke=False`, `device=DEVICE`를 명시하며 로드 후 실제 실행 장치와 모델 크기를 검사합니다.
- CUDA bf16 지원 시 bf16, 그 외 fp16을 사용합니다. Adapter 학습 가중치는 float32입니다.
- 378×378 letterbox 입력 → CLS 없는 729개 패치 → 4개씩 묶고 마지막 zero-padding → 183 visual tokens.
- 기본 LoRA rank는 64입니다. GPU 메모리 부족 시 배치/토큰 길이를 조정하거나 `LOAD_IN_4BIT=True`를 선택한 뒤 커널 재시작부터 진행하세요. 이 노트북은 GPU 메모리 용량만으로 실행 가능성을 보장하지 않습니다.
- 토큰 길이 초과 오류가 발생하면 데이터 길이를 확인하고 설정을 조정하세요. HTML을 자동으로 잘라내지 않습니다.

In [ ]:
model_config = ModelConfig(
    smoke=False,
    vision_model="naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B",
    language_model="codellama/CodeLlama-7b-hf",  # 로컬 가중치 폴더도 가능
    device=DEVICE,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    gradient_checkpointing=True,
    lora_r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    max_prompt_tokens=MAX_PROMPT_TOKENS,
    max_target_tokens=MAX_TARGET_TOKENS,
    seed=SEED,
)
seed_everything(SEED)
model = PosterLlama.from_config(model_config)
if model.config.smoke or model.device.type != "cuda":
    raise RuntimeError("실제 모델의 CUDA 로드에 실패했습니다. 커널을 재시작하고 설정을 확인하세요.")
summary = model.parameter_summary()
if summary["vision"]["total"] < 1_000_000 or summary["llm"]["total"] < 1_000_000:
    raise RuntimeError("대형 사전학습 모델이 아닌 소형 모델이 로드되었습니다.")
print("smoke:", model.config.smoke, "device:", model.device)
for name, counts in summary.items():
    print(f"{name}: total={counts['total']:,}, trainable={counts['trainable']:,}")
print("1단계 학습 대상:", model.assert_trainable_parameters())

RUN_DIR.mkdir(parents=True, exist_ok=False)
run_config = {
    "model": model_config.to_dict(),
    "stage1_training": vars(stage1_train_config),
    "stage2_training": vars(stage2_train_config),
    "train_stage1": TRAIN_STAGE1,
    "stage1_checkpoint": str(STAGE1_CHECKPOINT),
    "paper_task": PAPER_TASK,
    "datasets": {k: str(v) if v is not None else None for k, v in {
        "stage1": STAGE1_JSONL, "stage2": STAGE2_JSONL,
        "stage1_val": STAGE1_VAL_JSONL, "stage2_val": STAGE2_VAL_JSONL,
        "stage1_image_root": STAGE1_IMAGE_ROOT, "stage2_image_root": STAGE2_IMAGE_ROOT,
        "stage1_val_image_root": STAGE1_VAL_IMAGE_ROOT, "stage2_val_image_root": STAGE2_VAL_IMAGE_ROOT,
    }.items()},
}
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. 1단계 — 실제 이미지·캡션으로 Adapter 학습

정답 caption과 EOS에만 loss를 적용합니다. 비전/CodeLlama 기본 가중치는 고정하고, frozen LLM을 통과하는 gradient로 Adapter를 학습합니다. **전체 데이터에 대해 설정한 에폭 수만큼 실행**하며 1 step 제한은 없습니다.

기존 학습 함수는 매 에폭 끝에 해당 단계 폴더의 체크포인트와 `history.json`을 갱신합니다. 최종 파일은 마지막 완료 에폭의 가중치입니다. optimizer/scheduler/RNG 상태까지 복원하는 학습 재개나 best checkpoint 자동 선택은 지원하지 않습니다. 학습 중 커널/서버가 종료되면 마지막 완료 에폭 이후의 업데이트는 저장되지 않을 수 있습니다.

In [ ]:
if model.config.smoke or model.device.type != "cuda" or model.stage != 1:
    raise RuntimeError("1단계 실제 CUDA 모델이 필요합니다. 앞 셀부터 실행하세요.")
if TRAIN_STAGE1:
    stage1_history = train_stage(
        model, stage1_data, stage1_train_config,
        STAGE1_CHECKPOINT, validation=val1,
    )
    print("1단계 완료:", stage1_history[-1])
else:
    model.load_stage1_adapter(STAGE1_CHECKPOINT)
    print("기존 실제 HCX 1단계 Adapter를 불러왔습니다.")
print("1단계 체크포인트:", STAGE1_CHECKPOINT)

## 4. 2단계 — 실제 HTML 정답으로 CodeLlama LoRA 학습

1단계 Adapter를 다시 읽고 동결한 뒤 `q_proj`, `v_proj`의 LoRA만 학습합니다. 비전 타워와 CodeLlama 기본 가중치도 계속 동결됩니다. `STAGE2_EPOCHS`만큼 전체 데이터를 학습하고 마지막 완료 에폭의 Adapter/LoRA를 저장합니다.

In [ ]:
if model.config.smoke or model.device.type != "cuda" or model.stage != 1:
    raise RuntimeError("1단계를 마친 실제 CUDA 모델이 필요합니다. 2단계 셀을 중복 실행하지 마세요.")
model.load_stage1_adapter(STAGE1_CHECKPOINT)
model.start_stage2()
print(model.parameter_summary())
print("2단계 학습 파라미터 예:", model.assert_trainable_parameters()[:6])

stage2_history = train_stage(
    model, stage2_data, stage2_train_config,
    STAGE2_CHECKPOINT, validation=val2,
)
print("2단계 완료:", stage2_history[-1])
print("2단계 체크포인트:", STAGE2_CHECKPOINT)
print("학습 기록:", STAGE2_CHECKPOINT / "history.json")

## 5. 학습한 모델로 추론

기본적으로 방금 학습한 모델을 그대로 사용합니다. 저장·복원을 확인하려면 `RELOAD_FOR_INFERENCE=True`로 바꾸세요. 복원 전 기존 모델을 해제해서 대형 모델이 GPU에 중복 적재되지 않도록 합니다.

2단계 검증 데이터가 지정됐다면 그 중 한 장을 사용합니다. 검증 데이터가 없다면 학습 샘플을 사용하므로 이 출력은 별도 테스트 성능을 의미하지 않습니다. 생성 결과는 `generated.txt`에 저장하며, 잘못된 HTML은 아래 시각화 셀에서 오류로 보고합니다.

In [ ]:
RELOAD_FOR_INFERENCE = False
if RELOAD_FOR_INFERENCE:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    model = PosterLlama.from_checkpoint(
        STAGE2_CHECKPOINT, device=DEVICE, dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT, gradient_checkpointing=False,
    )
if model.config.smoke or model.device.type != "cuda" or model.stage != 2:
    raise RuntimeError("2단계 학습이 끝난 실제 CUDA 모델이 필요합니다.")
model.eval()
pixels = model.image_processor([example["image"]])
generated_html = model.generate(pixels, [example["prompt"]], max_new_tokens=MAX_TARGET_TOKENS)[0]
(RUN_DIR / "generated.txt").write_text(generated_html, encoding="utf-8")
print("예제 출처:", EXAMPLE_SOURCE)
print(generated_html)

## 6. 좌표 시각화

정답 레이아웃 미리보기와 모델 출력을 구분합니다. 렌더러는 `<svg>/<rect>` 좌표만 해석하고 원본 이미지에 박스를 표시합니다. 실제 홍보 문구의 폰트·색상·타이포그래피 배치는 별도 작업입니다. 생성 HTML이 잘못되거나 좌표가 캔버스를 벗어나면 오류를 표시합니다.

In [ ]:
print("정답 레이아웃 (모델 출력 아님)")
display(render_layout(example["image"], example["target"], RUN_DIR / "ground_truth.png"))
try:
    preview = render_layout(example["image"], generated_html, RUN_DIR / "generated_layout.png")
except ValueError as error:
    print("생성 레이아웃 검증 실패:", error)
    print("generated.txt와 validation loss를 확인하세요. 데이터·학습량·생성 길이 검토가 필요합니다.")
else:
    print("모델 생성 레이아웃")
    display(preview)

## 7. 내 이미지와 홍보 문구로 실행 (선택)

실학습을 마친 후 `CUSTOM_IMAGE`에 제품/배경 이미지 경로를 넣으세요. 원본 포스터의 글자가 제거된 이미지가 레이아웃 학습에 적합합니다. 이미지가 바뀌면 캔버스 크기도 실제 이미지 크기를 사용합니다.

In [ ]:
CUSTOM_IMAGE = None  # 예: PROJECT_ROOT / "data/my_product.jpg"
PROMO_TEXTS = ["신제품 출시", "지금 구매하면 20% 할인"]

if CUSTOM_IMAGE is not None:
    if model.config.smoke or model.device.type != "cuda" or model.stage != 2:
        raise RuntimeError("2단계 학습이 끝난 실제 CUDA 모델이 필요합니다.")
    with Image.open(CUSTOM_IMAGE) as source:
        custom_image = source.convert("RGB")
    prompt = build_layout_prompt(PROMO_TEXTS, width=custom_image.width, height=custom_image.height)
    result = model.generate(model.image_processor([custom_image]), [prompt], max_new_tokens=MAX_TARGET_TOKENS)[0]
    print(result)
    (RUN_DIR / "custom_generated.txt").write_text(result, encoding="utf-8")
    display(render_layout(custom_image, result, RUN_DIR / "custom_layout.png"))